#### Code to convert daily detected granules for PRR ocean color to an html file

In [1]:
# imports, make meng's tools available

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import sys
sys.path.append(r'C:\Users\gtrolley\Documents\GitHub\pace-rapid-response\dust\mapoltool\tools')
from detection_html_all import *
import io
from io import BytesIO
import os

In [ ]:
# function definitions

def fig_to_base64(input_source, factor=1, max_height=None, output_format='PNG', quality=100):
    '''function by gt
       A function to manipulate figures made by fig, ax = plt.subplots OR PNG file paths to prepare them for writing
       as base64 encoded strings to html files. Returns a base64 encoded string manipulated 
       with the desired factor and quality. factor can now be a float, factor=1.5 means size is reduced by 33%
       
       Args:
           input_source: Either a matplotlib figure object or a string path to a PNG file
           factor: Float resize factor (factor=1.5 means size is reduced by 33%, factor=2.0 means 50% reduction)
           max_height: If specified, resize image to this max height while maintaining aspect ratio
           output_format: 'PNG' or 'JPEG'
           quality: JPEG quality (1-100, ignored for PNG)
    '''
    
    # Check if input is a string (file path) or matplotlib figure
    if isinstance(input_source, str):
        # Handle file path input
        if not os.path.exists(input_source):
            raise FileNotFoundError(f"Image file not found: {input_source}")
        
        # Open the image directly from file
        img = Image.open(input_source)
        #print(f"Loaded image from file: {input_source}")
        original_size = img.size
        
    else:
        # Handle matplotlib figure input (original logic)
        buffer = io.BytesIO()
        input_source.savefig(buffer, format=output_format, dpi=300)
        buffer.seek(0)
        
        # Open image from buffer
        img = Image.open(buffer)
        original_size = img.size
    
    #print(f"Original size: {original_size}")
    
    # Resize image if factor > 1 (now supports floats!)
    if factor > 1:
        new_width = max(1, int(img.width / factor))    # Changed from // to / and added int()
        new_height = max(1, int(img.height / factor))  # Changed from // to / and added int()
        img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)
    
    # Additional resize based on max_height (this happens after factor resize)
    if max_height and img.height > max_height:
        aspect_ratio = img.width / img.height
        new_height = max_height
        new_width = int(max_height * aspect_ratio)
        img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)
    
    #print(f"Final size: {img.size}")

    # Save processed image to buffer
    output_buffer = io.BytesIO()
    
    if output_format.upper() == 'JPEG':
        img = img.convert("RGB")  # JPEG doesn't support transparency
        img.save(output_buffer, format=output_format, quality=quality)
    else:
        img.save(output_buffer, format=output_format)  # PNG ignores quality
    
    # Convert to base64
    base64_string = base64.b64encode(output_buffer.getvalue()).decode("utf-8")
    return base64_string


def write_html_image_row(imgs, capts, sizes, fill_width=True, equal_height=False):

    
    row_classes = ['row']
    if fill_width:
        row_classes.append('full-width')
    if equal_height:
        row_classes.append('equal-height-row')
    if len(imgs) == 1:  # Add single-image class for single images
        row_classes.append('single-image')
    
    f.write(f"<div class='{' '.join(row_classes)}'>\n")

    for i in range(len(imgs)):
        f.write(f"<div class='img-container {sizes[i]}'>\n")
        f.write(f"<img src='data:image/png;base64,{imgs[i]}' alt='{capts[i]}' />\n")
        f.write(f"<div class='caption'>{capts[i]}</div>\n</div>\n")
    
    f.write("</div>\n")

In [9]:
# list files in this directory
os.listdir()

['20250915', '20250916', 'daily_OC_HTML.ipynb']

In [10]:
# USER INPUTS:
day = '20250916' # choose the directory from os.listdir above with the day of data you want

In [11]:
ofilepath = day + '/html/OCI_chlor_a_anomaly_daily_'+day+'.html'
granule_folders  = [item for item in os.listdir(day+'/png/') if os.path.isdir(os.path.join(day+'/png/', item))]
header_imgs  = [item for item in os.listdir(day+'/png/') if item.lower().endswith('.png')]

In [12]:
header_imgs

['L3_Chl_20250916.png',
 'L3_Chl_20250916_bboxes.png',
 'L3_Chl_30dayMean_20250916.png',
 'L3_Chl_30dayMean_20250916_bboxes.png',
 'L3_Chl_Anomaly_20250916.png',
 'L3_Chl_Anomaly_20250916_bboxes.png']

In [63]:
# html file anatomy:

# overall header: provide oversight for method, and show: 
# daily vs monthly chla (row 1)
# chla anomaly, fille width figure

# then, granule by granule add figures


h1 = 'PACE OCI daily Chlorophyll-a Anomaly, '+day
tab_title = day+'_chla_anom_oci'
with open(ofilepath, "w") as f:
        # #Start the HTML file
        f.write("<!DOCTYPE html>\n<html>\n<head>\n<meta charset='UTF-8'>\n")
        f.write(f"<title>{tab_title}</title>\n")
        f.write("<style>\n")
        f.write("body { font-family: Arial, sans-serif; margin: 20px; }\n")
        f.write(".gallery { display: flex; flex-direction: column; gap: 20px; }\n")

        # Base row styles
        f.write(".row { display: flex; width: 100%; gap: 10px; align-items: flex-start; }\n")

        # Base size classes
        f.write(".small { flex: 1; }\n")
        f.write(".large { flex: 1; min-width: 0; }\n")

        # Container styles
        f.write(".img-container { display: flex; flex-direction: column; align-items: center; border: 1px solid #ccc; ")
        f.write("padding: 10px; border-radius: 10px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); background-color: #fafafa; height: min-content; }\n")

        # Image styles
        f.write(".img-container img { width: 100%; height: 300px; display: block; border-radius: 5px; object-fit: contain; }\n")
        f.write(".equal-height-row .img-container img { width: 100%; height: 300px; display: block; border-radius: 5px; object-fit: contain; }\n")

        # Full-width overrides (put AFTER base classes for higher specificity)
        f.write(".row.full-width.single-image { justify-content: center; }\n")
        f.write(".row.full-width.single-image .img-container { flex: 0 0 70%; max-width: 70%; }\n")
        #f.write(".row.full-width.single-image .img-container { flex: 1; max-width: 100%; }\n")
        #f.write(".row.full-width.single-image .img-container { flex: 1; }\n")
        f.write(".row.full-width.single-image .img-container img { width: 100%; height: auto; object-fit: fill; }\n")  # ADD THIS LINE


        f.write(".caption { margin-top: 10px; font-weight: bold; text-align: center; }\n")
        f.write("</style>\n</head>\n<body>\n")
                
        f.write(f"<h1 style='text-align: center; text-decoration: underline;'>{h1}</h1>\n")
        f.write(f"""<p>The Bloom Detection Dashboard identifies L2 PACE OCI granules that exhibit substantial
                 chlorophyll-a anomalies. These anomalies are calculated from L3 daily composites relative to
                 the preceding 30-day L3 composite mean (i.e., chlorophyll-a anomaly = daily chlorophyll-a - 
                30-day running chlorophyll-a mean). Potential bloom regions are detected in the L3 data by 
                partitioning the scene into 100x100 L3-pixel blocks and flagging blocks where at least 10% of 
                pixels show a chlorophyll-a anomaly greater than 1 mg/m^3. Flagged blocks serve as bounding boxes 
                to locate corresponding L2 granules for the same day. These L2 granules are shown with optical and 
                biogeochemical parameters overlaid on true-color imagery with embedded red squares to highlight the 
                L3 blocks that contain the large chlorophyll-a anomalies.</p>\n""")

        f.write(f"<div>Developed by Graham Trolley and Matthew Kehrli</div>\n")

        imgs = [fig_to_base64(day+'/png/'+'L3_Chl_'+day+'_bboxes.png'),fig_to_base64(day+'/png/'+'L3_Chl_30dayMean_'+day+'_bboxes.png') ]
        capts = ['Daily Chl_a '+day, 'Chl_a 30-day']
        sizes = ['large', 'large']
        write_html_image_row(imgs, capts, sizes)

        imgs = [fig_to_base64(day+'/png/'+'L3_Chl_Anomaly_'+day+'_bboxes.png') ]
        capts = ['Chl_a anomaly '+day+' vs 30-day mean']
        sizes = ['large']
        write_html_image_row(imgs, capts, sizes, fill_width=True, equal_height=False)# single-image row

        #header is done, now write a loop to plot all the granule data

        granule_images_shrink_factor = 1.7 # use a scale factor to reduce quality of saved images, greatly aids in keeping filesize reasonable
        for granule in granule_folders:
                #f.write(f"<div>______________________________________________________</div>\n")
                download_url = 'https://oceandata.sci.gsfc.nasa.gov/getfile/'+'PACE_OCI.'+granule+'.L2.OC_BGC.V3_1.nc'
                download_url_nrt = 'https://oceandata.sci.gsfc.nasa.gov/getfile/PACE_OCI.'+granule+'.L2.OC_BGC.V3_1.NRT.nc'
                f.write(f"<h2 style='text-align: center; text-decoration: underline;'>{'Granule '+granule}\n")
                f.write(f"<a href='{download_url}' class='title-link' target='_blank' title='Download Data'> Refined ⬇️</a>")
                f.write(f"<a href='{download_url_nrt}' class='title-link' target='_blank' title='Download Data'> NRT ⬇️ </a>")
                f.write("</h2>\n")
                #f.write(f"<div>{'NRT L2 link: ' + 'https://oceandata.sci.gsfc.nasa.gov/getfile/PACE_OCI.'+granule+'.L2.OC_BGC.V3_1.NRT.nc'}</div>\n")
                #f.write(f"<div>{'L2 link: ' + 'https://oceandata.sci.gsfc.nasa.gov/getfile/'+'PACE_OCI.'+granule+'.L2.OC_BGC.V3_1.nc'}</div>\n")

                carbon_phyto_im = fig_to_base64(day+'/png/'+granule+'/'+granule+'_carbon_phyto_overlay.png', factor = granule_images_shrink_factor)
                chlor_a_im = fig_to_base64(day+'/png/'+granule+'/'+granule+'_chlor_a_overlay.png', factor = granule_images_shrink_factor)
                poc_im = fig_to_base64(day+'/png/'+granule+'/'+granule+'_poc_overlay.png', factor = granule_images_shrink_factor)
                avw_im = fig_to_base64(day+'/png/'+granule+'/'+granule+'_avw_overlay.png', factor = granule_images_shrink_factor)
                nflh_im = fig_to_base64(day+'/png/'+granule+'/'+granule+'_nflh_overlay.png', factor = granule_images_shrink_factor)
                outline_im = fig_to_base64(day+'/png/'+granule+'/'+granule+'_outline.png', factor = granule_images_shrink_factor)


                imgs = [outline_im,avw_im,poc_im]
                capts = ['&nbsp;','AVW','POC'] #non-breaking space used to keep galleries aligned after adding captions for other plots
                sizes = ['large', 'large', 'large']
                write_html_image_row(imgs, capts, sizes)
                
                imgs = [chlor_a_im,nflh_im,carbon_phyto_im]
                capts = ['Chl-a','NFLH','Phyto Carbon']
                sizes = ['large', 'large', 'large']
                write_html_image_row(imgs, capts, sizes)

        f.write(f"<br>\n")  # Creates 3 line breaks
        
        f.write(f"<div>Granule download note: recent L2 granules need to use the NRT (near-real time) download link, older ones need to use the refined link. If one link isnt working, try the other. If both links aren't working, there was probably a data version update; you can manually correct the end of the filename to the correct version to fix the download link, i.e. at the end of the file, .L2.OC_BGC.V3_1.NRT.nc the V3_1 is changed to reflect the newest version</div>\n")
        f.write("</body>\n</html>\n")

In [9]:
day+'/png/'+'L3_Chl_'+day+'_bboxes.png'

'20250915/png/L3_Chl_20250915_bboxes.png'

In [25]:
os.listdir('20250916/png/20250916T173802')

['20250916T173802_avw_overlay.png',
 '20250916T173802_carbon_phyto_overlay.png',
 '20250916T173802_chlor_a_overlay.png',
 '20250916T173802_nflh_overlay.png',
 '20250916T173802_outline.png',
 '20250916T173802_poc_overlay.png']

In [15]:
# string anatomy to retrieve files:


#'https://oceandata.sci.gsfc.nasa.gov/getfile/'+'PACE_OCI.'+20251017T004249+'.L2.OC_BGC.V3_1.NRT.nc'

'https://oceandata.sci.gsfc.nasa.gov/getfile/'+'PACE_OCI.'+'20250915T004440'+'.L2.OC_BGC.V3_1.NRT.nc'


#PACE_OCI.20250829T000354.L2.OC_BGC.V3_1.nc
'https://oceandata.sci.gsfc.nasa.gov/getfile/'+'PACE_OCI.'+'20250829T000354'+'.L2.OC_BGC.V3_1.nc'

'https://oceandata.sci.gsfc.nasa.gov/getfile/PACE_OCI.20250829T000354.L2.OC_BGC.V3_1.nc'

In [10]:
os.listdir(day+'/png/'+granule_folders[0])

['20250915T003940_carbon_phyto_overlay.png',
 '20250915T003940_chlor_a_overlay.png',
 '20250915T003940_poc_overlay.png']

In [11]:
granule_folders[0]

'20250915T003940'

In [12]:
day+'/png/'+

SyntaxError: invalid syntax (3150687630.py, line 1)